# Alignment Quality Control with AlignmentViewer

This notebook demonstrates how to use the `AlignmentViewer` class to quickly visualize alignment quality directly from JSON.bz2 files without needing IGV or Geneious.

**Key Features:**
- Load alignments directly from JSON.bz2 files
- Generate IGV-like coverage plots
- Create interactive HTML reports
- Batch process multiple samples
- Fast QC for 64 mutant tRNAs across 256 samples

**Author:** Viewer-Specialist  
**Date:** 2026-02-10

## Setup

In [ ]:
# Standard imports
import os
import sys
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

# Add repo to path if needed
repo_path = Path.cwd().parent
if str(repo_path) not in sys.path:
    sys.path.insert(0, str(repo_path))

# Import AlignmentViewer
from trnaseq.visualization import AlignmentViewer

# Set matplotlib backend
%matplotlib inline

print(f"Repository path: {repo_path}")
print(f"Working directory: {Path.cwd()}")

## Example 1: Quick Coverage Check for a Single tRNA

The simplest use case - quickly visualize alignment coverage for a specific tRNA.

In [ ]:
# Path to your alignment JSON file
# Replace this with actual path from your project
json_file = repo_path / 'projects' / 'example' / 'data' / 'SWalign' / '100p_SWalign.json.bz2'

# Check if file exists (adjust path as needed for your data)
if json_file.exists():
    print(f"Found alignment file: {json_file.name}")
else:
    print(f"File not found: {json_file}")
    print("Please adjust the path to point to your alignment JSON file")

In [ ]:
# Create viewer instance
viewer = AlignmentViewer(json_file)

# List available tRNAs in the file
trna_list = viewer.list_trnas(min_reads=10)
print(f"\nTop 10 tRNAs by read count:")
print(trna_list.head(10))

In [ ]:
# Select a tRNA to visualize (use first one from the list)
if len(trna_list) > 0:
    selected_trna = trna_list.iloc[0]['tRNA']
    print(f"Visualizing: {selected_trna}")
    
    # Generate coverage plot
    fig = viewer.plot_coverage(selected_trna)
else:
    print("No tRNAs found with sufficient reads")

## Example 2: Generate Quick View (PNG + HTML)

The `quick_view()` method generates both a PNG coverage plot and an interactive HTML report in one command.

In [ ]:
# Generate both PNG and HTML reports
if len(trna_list) > 0:
    selected_trna = trna_list.iloc[0]['tRNA']
    png_path, html_path = viewer.quick_view(selected_trna)
    
    print(f"\nGenerated files:")
    print(f"  Coverage plot: {png_path}")
    print(f"  HTML report:   {html_path}")
    print(f"\nOpen {html_path} in your browser to view the interactive report!")

## Example 3: Detailed Coverage Analysis

Calculate coverage statistics and identify positions with high mismatch rates.

In [ ]:
# Calculate coverage for a specific tRNA
if len(trna_list) > 0:
    selected_trna = trna_list.iloc[0]['tRNA']
    cov_df = viewer.calculate_coverage(selected_trna)
    
    print(f"\nCoverage statistics for {selected_trna}:")
    print(f"  Total positions: {len(cov_df)}")
    print(f"  Mean coverage: {cov_df['coverage'].mean():.1f}")
    print(f"  Max coverage: {cov_df['coverage'].max()}")
    print(f"  Mean mismatch rate: {cov_df['mismatch_rate'].mean():.2f}%")
    
    # Find positions with high mismatch rates
    high_mismatch = cov_df[cov_df['mismatch_rate'] > 5.0]
    if len(high_mismatch) > 0:
        print(f"\nPositions with >5% mismatch rate:")
        print(high_mismatch[['position', 'coverage', 'mismatches', 'mismatch_rate']])
    else:
        print(f"\nNo positions with >5% mismatch rate (good alignment quality!)")

## Example 4: Compare Multiple tRNAs

Generate coverage plots for multiple tRNAs to compare alignment quality.

In [ ]:
# Select top 3 tRNAs by read count
if len(trna_list) >= 3:
    top_trnas = trna_list.head(3)['tRNA'].tolist()
    
    # Create a comparison plot
    fig, axes = plt.subplots(len(top_trnas), 1, figsize=(14, 4*len(top_trnas)))
    
    for idx, trna in enumerate(top_trnas):
        cov_df = viewer.calculate_coverage(trna)
        ax = axes[idx] if len(top_trnas) > 1 else axes
        
        # Plot coverage
        ax.bar(cov_df['position'], cov_df['coverage'], 
               color='gray', alpha=0.7, label='Coverage')
        ax.bar(cov_df['position'], cov_df['mismatches'],
               color='red', alpha=0.8, label='Mismatches')
        
        ax.set_ylabel('Read Count')
        ax.set_title(f'{trna} (n={trna_list[trna_list["tRNA"]==trna]["reads"].values[0]} reads)')
        ax.legend()
        ax.grid(axis='y', alpha=0.3)
        
        if idx == len(top_trnas) - 1:
            ax.set_xlabel('Position on tRNA')
    
    plt.tight_layout()
    plt.savefig('top3_trna_coverage_comparison.png', dpi=300, bbox_inches='tight')
    print("Saved: top3_trna_coverage_comparison.png")
else:
    print("Not enough tRNAs for comparison")

## Example 5: Batch Processing Multiple Samples

Process multiple alignment files to generate QC reports for all samples.

In [ ]:
# Define directory containing alignment files
align_dir = repo_path / 'projects' / 'example' / 'data' / 'SWalign'

# Find all alignment JSON files
if align_dir.exists():
    json_files = list(align_dir.glob('*_SWalign.json.bz2'))
    print(f"Found {len(json_files)} alignment files in {align_dir.name}/")
    
    for json_file in json_files[:3]:  # Process first 3 files as example
        print(f"\n  - {json_file.name}")
else:
    print(f"Directory not found: {align_dir}")
    print("Adjust the path to match your project structure")

In [ ]:
# Batch generate reports for a specific tRNA across multiple samples
target_trna = 'tRNA-Ala-TGC-1'  # Change to your target tRNA

if align_dir.exists():
    json_files = list(align_dir.glob('*_SWalign.json.bz2'))
    
    # Create output directory for QC reports
    qc_dir = Path('qc_reports')
    qc_dir.mkdir(exist_ok=True)
    
    results = []
    
    for json_file in json_files[:3]:  # Process first 3 as example
        try:
            viewer = AlignmentViewer(json_file)
            
            # Try to find the target tRNA or use top tRNA
            trna_list = viewer.list_trnas(min_reads=10)
            
            if len(trna_list) > 0:
                # Use target tRNA if available, otherwise use first one
                if target_trna in trna_list['tRNA'].values:
                    selected_trna = target_trna
                else:
                    selected_trna = trna_list.iloc[0]['tRNA']
                
                # Generate coverage plot
                sample_name = json_file.stem.replace('_SWalign', '')
                output_png = qc_dir / f"{sample_name}_{selected_trna.replace('/', '-')}_coverage.png"
                
                viewer.plot_coverage(selected_trna, output=str(output_png))
                
                # Calculate summary stats
                cov_df = viewer.calculate_coverage(selected_trna)
                results.append({
                    'sample': sample_name,
                    'tRNA': selected_trna,
                    'mean_coverage': cov_df['coverage'].mean(),
                    'max_coverage': cov_df['coverage'].max(),
                    'mean_mismatch_rate': cov_df['mismatch_rate'].mean()
                })
        except Exception as e:
            print(f"Error processing {json_file.name}: {e}")
    
    # Create summary DataFrame
    if results:
        summary_df = pd.DataFrame(results)
        print(f"\nBatch processing summary:")
        print(summary_df)
        
        # Save summary
        summary_df.to_csv(qc_dir / 'qc_summary.csv', index=False)
        print(f"\nSaved summary to {qc_dir / 'qc_summary.csv'}")
else:
    print("Skipping batch processing - no alignment files found")

## Example 6: Troubleshooting Alignment Issues

Use the HTML report to investigate alignment quality in detail.

In [ ]:
# Generate detailed HTML report for troubleshooting
if len(trna_list) > 0:
    selected_trna = trna_list.iloc[0]['tRNA']
    
    # Get alignment details
    details = viewer.get_alignment_details(selected_trna, max_reads=20)
    
    print(f"\nAlignment details for {selected_trna}:")
    print(f"  Total reads retrieved: {len(details)}")
    
    if len(details) > 0:
        # Show first alignment as example
        first_aln = details[0]
        print(f"\nExample alignment:")
        print(f"  Read ID: {first_aln['read_id'][:50]}...")
        print(f"  Score: {first_aln['score']}")
        print(f"  Ref position: {first_aln['dpos'][0]}-{first_aln['dpos'][1]}")
        print(f"  Fmax score: {first_aln['Fmax_score']:.3f}")
        print(f"\n  Query: {first_aln['qseq']}")
        print(f"         {first_aln['aseq']}")
        print(f"  Ref:   {first_aln['dseq']}")
    
    # Generate full HTML report
    html_path = viewer.create_html_report(selected_trna, 
                                          output=f'{selected_trna.replace("/", "-")}_detailed_report.html')
    print(f"\nDetailed HTML report saved to: {html_path}")

## Example 7: Quality Metrics Dashboard

Create a summary dashboard showing alignment quality across multiple tRNAs.

In [ ]:
# Analyze quality metrics for all tRNAs
if len(trna_list) > 0:
    quality_metrics = []
    
    # Analyze top 10 tRNAs
    for trna_name in trna_list.head(10)['tRNA']:
        cov_df = viewer.calculate_coverage(trna_name)
        
        # Calculate quality metrics
        quality_metrics.append({
            'tRNA': trna_name,
            'reads': trna_list[trna_list['tRNA']==trna_name]['reads'].values[0],
            'length': len(cov_df),
            'mean_coverage': cov_df['coverage'].mean(),
            'coverage_stddev': cov_df['coverage'].std(),
            'mean_mismatch_rate': cov_df['mismatch_rate'].mean(),
            'max_mismatch_rate': cov_df['mismatch_rate'].max(),
            'positions_above_5pct_mismatch': (cov_df['mismatch_rate'] > 5.0).sum()
        })
    
    # Create quality dashboard
    quality_df = pd.DataFrame(quality_metrics)
    
    print("\nAlignment Quality Dashboard:")
    print("="*80)
    print(quality_df.to_string(index=False))
    
    # Save to file
    quality_df.to_csv('alignment_quality_metrics.csv', index=False)
    print("\nSaved quality metrics to: alignment_quality_metrics.csv")
    
    # Create visualization
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Plot 1: Mean mismatch rate by tRNA
    axes[0].barh(range(len(quality_df)), quality_df['mean_mismatch_rate'])
    axes[0].set_yticks(range(len(quality_df)))
    axes[0].set_yticklabels([t[:30] for t in quality_df['tRNA']], fontsize=8)
    axes[0].set_xlabel('Mean Mismatch Rate (%)')
    axes[0].set_title('Mean Mismatch Rate by tRNA')
    axes[0].grid(axis='x', alpha=0.3)
    
    # Plot 2: Coverage vs mismatch rate
    axes[1].scatter(quality_df['mean_coverage'], quality_df['mean_mismatch_rate'],
                   s=quality_df['reads']/10, alpha=0.6)
    axes[1].set_xlabel('Mean Coverage')
    axes[1].set_ylabel('Mean Mismatch Rate (%)')
    axes[1].set_title('Coverage vs Mismatch Rate\n(bubble size = read count)')
    axes[1].grid(alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('alignment_quality_dashboard.png', dpi=300, bbox_inches='tight')
    print("Saved dashboard plot to: alignment_quality_dashboard.png")
else:
    print("No tRNAs found for quality analysis")

## Summary

The `AlignmentViewer` class provides a fast, lightweight way to visualize alignment quality without external tools. Key advantages:

1. **Fast**: Reads JSON.bz2 files directly, no conversion needed
2. **Lightweight**: Only matplotlib and pandas dependencies
3. **Flexible**: Works with single tRNAs or batch processes hundreds of samples
4. **Publication-ready**: Generates high-quality PNG plots
5. **Shareable**: Creates standalone HTML reports

For the PJ39 project with 256 samples and 64 mutant tRNAs, this tool enables rapid QC to identify alignment issues before downstream analysis.

## Next Steps

1. Adapt the paths in this notebook to your project structure
2. Run QC on your alignment files
3. Identify tRNAs with low coverage or high mismatch rates
4. Generate HTML reports for sharing with collaborators
5. Use quality metrics to filter data for downstream analysis